# Data Profiling in skforecast-ai

The **Data Profiling** stage is the first and arguably most critical step in the `skforecast-ai` pipeline. Before any forecasting logic is generated, the assistant exhaustively understands the shape, quality, and limitations of the data.

It acts as an **immutable metadata extractor**, returning a strictly typed `DataProfile` object.

In this guide, we will walk through how to use the `.profile()` method for single-series and multi-series datasets, and see how it handles missing values and warnings.

In [24]:
import pandas as pd
import numpy as np
from pprint import pprint
from skforecast_ai import ForecastingAssistant

## 1. Single Series Profiling

Let's create a synthetic daily sales dataset with an exogenous variable and some missing values to see how the profiler reacts.

In [25]:
# Create synthetic single-series data
dates = pd.date_range(start="2023-01-01", periods=100, freq="D")
np.random.seed(42)
sales = np.random.normal(50, 10, size=100)

# Introduce some missing values
sales[20:25] = np.nan

df_single = pd.DataFrame({
    "date": dates,
    "sales": sales,
    "temperature": np.random.normal(20, 5, size=100)
})

display(df_single.head())

,date,sales,temperature
0,2023-01-01,54.967142,12.923146
1,2023-01-02,48.617357,17.896773
2,2023-01-03,56.476885,18.286427
3,2023-01-04,65.230299,15.988614
4,2023-01-05,47.658466,19.193571


In [26]:
# Instantiate the assistant
assistant = ForecastingAssistant()

# Profile the data
profile = assistant.profile(
    data=df_single,
    target="sales",
    date_column="date"
)

The `profile()` method returns a `ForecastingProfile` which contains the `DataProfile`. Let's inspect the extracted metadata. 
Notice how it automatically detected the frequency (`D`), the exogenous variable (`temperature`), and the missing values in our target.

In [27]:
# Display key profiling results
data_profile = profile.data_profile
print(data_profile.model_dump_json(indent=2))

{
  "data_format": "single",
  "n_series": 1,
  "n_observations": 100,
  "series_lengths": null,
  "target": "sales",
  "target_dtype": "numeric",
  "target_stats": {
    "sales": {
      "min": 23.802548959102555,
      "max": 68.52278184508938,
      "mean": 48.97653457875855,
      "std": 9.062759944140469
    }
  },
  "missing_target": {
    "sales": 5
  },
  "date_column": "date",
  "series_id_column": null,
  "index_type": "datetime",
  "frequency": "D",
  "frequency_is_set": false,
  "index_is_monotonic": true,
  "has_gaps": false,
  "has_duplicate_timestamps": false,
  "exog_columns": [
    "temperature"
  ],
  "categorical_exog": [],
  "missing_exog": {},
  "data_path": "data.csv",
  "start_date": "2023-01-01",
  "end_train": "2023-03-21",
  "warnings": []
}


## 2. Multi-Series (Long Format)

When dealing with multiple time series stacked in a single DataFrame (long format), we use the `series_id_column` parameter. 
The profiler will smartly isolate individual timelines to verify frequencies and start dates across the entire dataset.

In [6]:
# Create synthetic long-format data
store_1 = pd.DataFrame({
    "date": pd.date_range(start="2023-01-01", periods=50, freq="D"),
    "store_id": "store_1",
    "sales": np.random.normal(100, 15, size=50)
})

store_2 = pd.DataFrame({
    "date": pd.date_range(start="2023-01-15", periods=36, freq="D"), # Starts later
    "store_id": "store_2",
    "sales": np.random.normal(80, 10, size=36)
})

df_long = pd.concat([store_1, store_2], ignore_index=True)
display(df_long.head())

,date,store_id,sales
0,2023-01-01,store_1,105.366810
1,2023-01-02,store_1,108.411768
2,2023-01-03,store_1,116.245769
3,2023-01-04,store_1,115.807031
4,2023-01-05,store_1,79.334959


In [28]:
profile = assistant.profile(
    data=df_long,
    target="sales",
    date_column="date",
    series_id_column="store_id"
)

data_profile = profile.data_profile
print(data_profile.model_dump_json(indent=2))

{
  "data_format": "long",
  "n_series": 2,
  "n_observations": 36,
  "series_lengths": {
    "store_1": 50,
    "store_2": 36
  },
  "target": "sales",
  "target_dtype": "numeric",
  "target_stats": {
    "store_1": {
      "min": 69.6228612001359,
      "max": 157.7909723598208,
      "mean": 102.2645401012103,
      "std": 16.379085607844807
    },
    "store_2": {
      "min": 47.58732659930927,
      "max": 101.33033374656267,
      "mean": 79.15936762328045,
      "std": 12.216422362378534
    }
  },
  "missing_target": {},
  "date_column": "date",
  "series_id_column": "store_id",
  "index_type": "datetime",
  "frequency": "D",
  "frequency_is_set": false,
  "index_is_monotonic": true,
  "has_gaps": false,
  "has_duplicate_timestamps": false,
  "exog_columns": [],
  "categorical_exog": [],
  "missing_exog": {},
  "data_path": "data.csv",
  "start_date": "2023-01-15",
  "end_train": "2023-02-09",
  "warnings": [
    "Short series: only 36 observations. Results may be unreliable w

## 3. Multi-Series (Wide Format)

If your series are organized in columns (wide format), simply pass a list of target column names.

In [29]:
df_wide = pd.DataFrame({
    "date": pd.date_range(start="2023-01-01", periods=60, freq="D"),
    "store_1_sales": np.random.normal(100, 15, size=60),
    "store_2_sales": np.random.normal(80, 10, size=60)
})

profile = assistant.profile(
    data=df_wide,
    target=["store_1_sales", "store_2_sales"],
    date_column="date"
)

data_profile = profile.data_profile
print(data_profile.model_dump_json(indent=2))

{
  "data_format": "wide",
  "n_series": 2,
  "n_observations": 60,
  "series_lengths": {
    "store_1_sales": 60,
    "store_2_sales": 60
  },
  "target": [
    "store_1_sales",
    "store_2_sales"
  ],
  "target_dtype": "numeric",
  "target_stats": {
    "store_1_sales": {
      "min": 69.6228612001359,
      "max": 157.7909723598208,
      "mean": 102.53353939516023,
      "std": 16.489573600695522
    },
    "store_2_sales": {
      "min": 47.58732659930927,
      "max": 101.33033374656267,
      "mean": 80.31059889487213,
      "std": 9.477693103389244
    }
  },
  "missing_target": {},
  "date_column": "date",
  "series_id_column": null,
  "index_type": "datetime",
  "frequency": "D",
  "frequency_is_set": false,
  "index_is_monotonic": true,
  "has_gaps": false,
  "has_duplicate_timestamps": false,
  "exog_columns": [],
  "categorical_exog": [],
  "missing_exog": {},
  "data_path": "data.csv",
  "start_date": "2023-01-01",
  "end_train": "2023-02-17",
  "warnings": []
}


## Conclusion

By rigorously profiling the data first, `skforecast-ai` achieves **fail-fast** mechanics (catching issues like constant targets immediately) and provides **deterministic routing** for the downstream recommendation engine, all while keeping the LLM focused on a pristine schema rather than noisy raw rows.